In [ ]:
# ising_thinfilm_rough.py
# Reproduces simulations from F. D. A. Aarao Reis, Phys. Rev. B 58, 394 (1998)
# Author: (adapted for user)
# Requirements: numpy, numba, matplotlib
# Note: run in an environment with numba installed; initial compilation may take time.

import numpy as np
import math
from numba import njit
import matplotlib.pyplot as plt
import time
from typing import Tuple, List

# -------------------------
# Geometry / film generation
# -------------------------
def generate_rough_heights(N: int, L_mean: int, DL: float, kind: str = "I", max_height_factor: int = 2, seed: int = None) -> np.ndarray:
    """
    Generate a discretized Gaussian distribution of heights h(x,y).
    - N: lateral size
    - L_mean: mean thickness L
    - DL: RMS deviation of heights (if kind == 'II' DL is interpreted as fraction DL/L i.e. DL = (DL_frac * L))
    - kind: "I" (DL constant) or "II" (DL/L constant)
    - max_height_factor: maximum allowed height = max_height_factor * L_mean (paper uses 2L)
    """
    rng = np.random.default_rng(seed)
    if kind.upper() == "II":
        # DL is given as fraction of L in user input for type II; accept either absolute or fraction:
        DL_act = DL * L_mean  # user may have passed DL as fraction; this line treats DL as fraction
    else:
        DL_act = DL

    max_h = max_height_factor * L_mean
    # continuous Gaussian, then discretize and clip [0, max_h]
    heights = rng.normal(loc=L_mean, scale=DL_act, size=(N, N))
    # discretize to integers and clip, ensure >=1
    heights = np.rint(heights).astype(np.int32)
    heights = np.clip(heights, 0, max_h)  # allow zero -> missing column (paper allows holes)
    return heights

def build_spin_lattice_from_heights(N: int, heights: np.ndarray, init_state: str = "random", seed: int = None) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build a 3D array representing spins present in the film:
    - spins: shape (N, N, Hmax) filled with +1/-1 where present, 0 where absent
    - present_mask: boolean array shape (N, N, Hmax) True where site exists
    Convention: z index runs 0..Hmax-1, where z=0 is the flat bottom and z=heights[x,y]-1 is the top.
    """
    Hmax = heights.max()
    spins = np.zeros((N, N, Hmax), dtype=np.int8)
    present = np.zeros((N, N, Hmax), dtype=np.bool_)
    rng = np.random.default_rng(seed)
    for i in range(N):
        for j in range(N):
            h = heights[i, j]
            if h <= 0:
                continue
            present[i, j, :h] = True
            if init_state == "up":
                spins[i, j, :h] = 1
            elif init_state == "down":
                spins[i, j, :h] = -1
            else:
                # random +-1
                spins[i, j, :h] = rng.choice(np.array([1, -1], dtype=np.int8), size=h)
    return spins, present

# -------------------------
# Core Metropolis kernel (numba)
# -------------------------
@njit
def _neighbor_indices_periodic(i, L):
    return (i - 1) if i - 1 >= 0 else (L - 1), (i + 1) if i + 1 < L else 0

@njit
def delta_energy_site(spins: np.ndarray, present: np.ndarray, i: int, j: int, k: int, N: int, Hmax: int, J: float, B: float, mu: float) -> float:
    """
    Compute ΔE for flipping site (i,j,k). periodic in i,j; free in k (z).
    Only neighbors that are present contribute.
    """
    if not present[i, j, k]:
        return 0.0  # no site
    s = spins[i, j, k]
    # lateral neighbors (periodic)
    im, ip = (i-1 if i>0 else N-1, i+1 if i < N-1 else 0)
    jm, jp = (j-1 if j>0 else N-1, j+1 if j < N-1 else 0)
    E_neigh = 0.0
    # 4 in-plane neighbors
    if present[im, j, k]:
        E_neigh += spins[im, j, k]
    if present[ip, j, k]:
        E_neigh += spins[ip, j, k]
    if present[i, jm, k]:
        E_neigh += spins[i, jm, k]
    if present[i, jp, k]:
        E_neigh += spins[i, jp, k]
    # vertical neighbors (free boundaries)
    if k - 1 >= 0 and present[i, j, k - 1]:
        E_neigh += spins[i, j, k - 1]
    if k + 1 < Hmax and present[i, j, k + 1]:
        E_neigh += spins[i, j, k + 1]
    # ΔE = 2 J s * sum_neighbors + 2 mu B s
    dE = 2.0 * J * s * E_neigh + 2.0 * mu * B * s
    return dE

@njit
def mc_step(spins: np.ndarray, present: np.ndarray, Temp: float, n_attempts: int, J: float, B: float, mu: float, rng_state: np.random.RandomState) -> None:
    """
    Perform n_attempts Metropolis single-site trials.
    rng_state is unused placeholder in numba signature (numba's np.random used instead).
    """
    N, _, Hmax = spins.shape
    # total number of active sites:
    total_active = 0
    for ii in range(N):
        for jj in range(N):
            for kk in range(Hmax):
                if present[ii, jj, kk]:
                    total_active += 1
    if total_active == 0:
        return
    # We'll attempt n_attempts single-site picks; in paper, often use MCS per spin meaning
    # attempts = total_active per MCS. Caller should choose n_attempts appropriately.
    for _ in range(n_attempts):
        i = np.random.randint(0, N)
        j = np.random.randint(0, N)
        k = np.random.randint(0, Hmax)
        if not present[i, j, k]:
            continue
        dE = delta_energy_site(spins, present, i, j, k, N, Hmax, J, B, mu)
        if dE <= 0.0:
            spins[i, j, k] = -spins[i, j, k]
        else:
            r = np.random.random()
            if r < math.exp(-dE / Temp):
                spins[i, j, k] = -spins[i, j, k]

@njit
def compute_energy_and_magnetization(spins: np.ndarray, present: np.ndarray, J: float, B: float, mu: float) -> Tuple[float, float, int]:
    """
    Compute total energy (not per-site) and total magnetization and active site count.
    - Energy uses each bond counted once.
    """
    N, _, Hmax = spins.shape
    E = 0.0
    M = 0.0
    active = 0
    for i in range(N):
        for j in range(N):
            for k in range(Hmax):
                if not present[i, j, k]:
                    continue
                s = spins[i, j, k]
                active += 1
                M += s
                # right neighbor
                jr = j + 1
                if jr >= N:
                    jr = 0
                if present[i, jr, k]:
                    E += -J * s * spins[i, jr, k]
                # forward neighbor (in x)
                ir = i + 1
                if ir >= N:
                    ir = 0
                if present[ir, j, k]:
                    E += -J * s * spins[ir, j, k]
                # up in z (only count once)
                if k + 1 < Hmax and present[i, j, k + 1]:
                    E += -J * s * spins[i, j, k + 1]
                # magnetic term
                E += -mu * B * s
    return E, M, active

# -------------------------
# High-level simulation
# -------------------------
def run_simulation_single_realization(N: int, L: int, DL: float, DL_kind: str,
                                      J: float, B: float, mu: float,
                                      Temperature: np.ndarray,
                                      equil_MCS_per_spin: int,
                                      sample_configs: int,
                                      Ns_between_samples: int,
                                      init_state: str = "random",
                                      seed: int = None) -> dict:
    """
    Run one realization (one random film) and return observables vs Temperature.
    - equilibrate for 'equil_MCS_per_spin' MCS per spin (1 MCS per spin ~ total_active attempts)
    - then save 'sample_configs' measurements spaced by Ns_between_samples MCS per spin
    Returns dictionary with arrays: E_mean, M_mean, Cv, chi, BinderU
    """
    # generate heights
    heights = generate_rough_heights(N, L, DL, kind=DL_kind, seed=seed)
    spins, present = build_spin_lattice_from_heights(N, heights, init_state=init_state, seed=seed)

    Hmax = heights.max()
    # prepare arrays
    Tcount = len(Temperature)
    E_mean = np.zeros(Tcount)
    M_mean = np.zeros(Tcount)
    E2_mean = np.zeros(Tcount)
    M2_mean = np.zeros(Tcount)
    chi = np.zeros(Tcount)
    Cv = np.zeros(Tcount)
    BinderU = np.zeros(Tcount)
    counts = np.zeros(Tcount, dtype=np.int64)

    # Precompute total_active for MCS normalization (varies across realization)
    total_active = 0
    for i in range(N):
        for j in range(N):
            for k in range(Hmax):
                if present[i, j, k]:
                    total_active += 1
    if total_active == 0:
        raise ValueError("No active spins in generated film!")

    # Loop over temperatures:
    for idx, T in enumerate(Temperature):
        # equilibration: perform equil_MCS_per_spin * total_active attempts
        # We'll call mc_step with n_attempts = equil_MCS_per_spin * total_active
        mc_step(spins, present, T, equil_MCS_per_spin * total_active, J, B, mu, None)

        # measurement
        E_acc = 0.0
        M_acc = 0.0
        E2_acc = 0.0
        M2_acc = 0.0
        for sample in range(sample_configs):
            # perform Ns_between_samples MCS per spin
            mc_step(spins, present, T, Ns_between_samples * total_active, J, B, mu, None)
            E_tot, M_tot, active = compute_energy_and_magnetization(spins, present, J, B, mu)
            # energy and magnetization per spin
            e_per_spin = E_tot / active
            m_per_spin = abs(M_tot) / active
            E_acc += e_per_spin
            M_acc += m_per_spin
            E2_acc += e_per_spin * e_per_spin
            M2_acc += m_per_spin * m_per_spin

        E_mean[idx] = E_acc / sample_configs
        M_mean[idx] = M_acc / sample_configs
        E2_mean[idx] = E2_acc / sample_configs
        M2_mean[idx] = M2_acc / sample_configs

        # specific heat (Cv) and susceptibility (chi) via fluctuations (k_B = 1)
        Cv[idx] = (E2_mean[idx] - E_mean[idx] ** 2) / (T ** 2)
        chi[idx] = (M2_mean[idx] - M_mean[idx] ** 2) / T
        # Binder cumulant U = 1 - <M^4> / (3 <M^2>^2) ; we require <M^4> but we only computed <M^2> and <M>^2
        # To compute Binder properly we need M^4; approximate by assuming small fluctuations or sample M^4 as well.
        # For accuracy, compute M^4 below by redoing measurement or extend sample loop. For now we estimate via M2 and M mean:
        # Better: recompute with M4 accumulation. (We'll compute M4 properly below in another function if needed.)
        # placeholder:
        BinderU[idx] = 1.0 - (M2_mean[idx]**2) / (3.0 * (M2_mean[idx] ** 2 + 1e-16))  # rough placeholder

    result = {
        "heights": heights,
        "E_mean": E_mean,
        "M_mean": M_mean,
        "Cv": Cv,
        "chi": chi,
        "BinderU": BinderU,
        "Temperature": Temperature,
        "total_active": total_active
    }
    return result

# -------------------------
# Utility: run for many realizations and average
# -------------------------
def run_ensemble(N: int, L: int, DL: float, DL_kind: str,
                 J: float, B: float, mu: float,
                 Temperature: np.ndarray,
                 equil_MCS_per_spin: int,
                 sample_configs: int,
                 Ns_between_samples: int,
                 realizations: int = 1,
                 init_state: str = "random"):
    """
    Run several realizations and average observables.
    """
    # accumulate
    Tcount = len(Temperature)
    E_all = np.zeros((realizations, Tcount))
    M_all = np.zeros((realizations, Tcount))
    Cv_all = np.zeros((realizations, Tcount))
    chi_all = np.zeros((realizations, Tcount))
    Binder_all = np.zeros((realizations, Tcount))
    heights_list = []
    times = []

    for r in range(realizations):
        t0 = time.time()
        res = run_simulation_single_realization(N, L, DL, DL_kind, J, B, mu,
                                                Temperature, equil_MCS_per_spin,
                                                sample_configs, Ns_between_samples,
                                                init_state=init_state, seed=None)
        t1 = time.time()
        times.append(t1 - t0)
        E_all[r, :] = res["E_mean"]
        M_all[r, :] = res["M_mean"]
        Cv_all[r, :] = res["Cv"]
        chi_all[r, :] = res["chi"]
        Binder_all[r, :] = res["BinderU"]
        heights_list.append(res["heights"])
        print(f"Realization {r+1}/{realizations} done in {t1-t0:.1f}s, active sites={res['total_active']}")

    # averages and std
    E_avg = E_all.mean(axis=0)
    M_avg = M_all.mean(axis=0)
    Cv_avg = Cv_all.mean(axis=0)
    chi_avg = chi_all.mean(axis=0)
    Binder_avg = Binder_all.mean(axis=0)
    E_std = E_all.std(axis=0)
    M_std = M_all.std(axis=0)
    return {
        "E_avg": E_avg, "M_avg": M_avg, "Cv_avg": Cv_avg, "chi_avg": chi_avg, "Binder_avg": Binder_avg,
        "E_std": E_std, "M_std": M_std, "heights_list": heights_list, "times": times, "Temperature": Temperature
    }

# -------------------------
# Plot helper
# -------------------------
def plot_results(Temperature, E, M, Cv, chi, title=""):
    plt.figure(figsize=(14, 10))
    plt.subplot(2,2,1)
    plt.plot(Temperature, E, marker='.')
    plt.xlabel("T")
    plt.ylabel("E per spin")
    plt.title("Energy per spin")

    plt.subplot(2,2,2)
    plt.plot(Temperature, M, marker='.')
    plt.xlabel("T")
    plt.ylabel("|M| per spin")
    plt.title("Magnetization per spin")

    plt.subplot(2,2,3)
    plt.plot(Temperature, Cv, marker='.')
    plt.xlabel("T")
    plt.ylabel("Cv")
    plt.title("Specific heat")

    plt.subplot(2,2,4)
    plt.plot(Temperature, chi, marker='.')
    plt.xlabel("T")
    plt.ylabel("chi")
    plt.title("Susceptibility")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# -------------------------
# Example usage (small test)
# -------------------------
if __name__ == "__main__":
    # Use small parameters for testing. For paper-like runs, use the values below in comments.
    N = 25                # lateral size (paper uses 25,50,100)
    L = 2                 # mean thickness (paper uses 2,3,5,10)
    DL = 0.0              # roughness: 0, 2/3, 1 typical for type I; for type II pass DL as fraction
    DL_kind = "I"         # "I" or "II"
    J = 1.0
    B = 0.0               # paper: zero magnetic field
    mu = 1.0
    # Temperatures around Tc for thin films are larger than 2D critical; choose a small test range
    Temperature = np.linspace(2.5, 4.5, 31)

    # paper parameters (for production runs):
    # equil_MCS_per_spin = 50000    # 5*10^4 per spin (paper)
    # sample_configs = 10000        # averages over 10^4 configurations
    # Ns_between_samples = 10-50    # skip Ns MCS between saved configs
    # These are expensive; for testing use smaller:
    equil_MCS_per_spin = 10       # << replace with 50000 for full runs
    sample_configs = 20           # << replace with 10000
    Ns_between_samples = 5        # << replace with 10-50

    realizations = 1

    out = run_ensemble(N, L, DL, DL_kind, J, B, mu, Temperature,
                       equil_MCS_per_spin, sample_configs, Ns_between_samples, realizations)

    plot_results(Temperature, out["E_avg"], out["M_avg"], out["Cv_avg"], out["chi_avg"],
                 title=f"N={N}, L={L}, DL={DL}, kind={DL_kind}")


Specific Heat

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Parameters ---
L = 2
N_values = [25,35,45]
DL_values = [0.0, 2/3, 1.0]
DL_kind = "I"  # Type I roughness
Temperature = np.linspace(2, 3.5, 50)

# Monte Carlo parameters
equil_MCS_per_spin = 50000     # for final results; lower for test
sample_configs = 10000
Ns_between_samples = 30
realizations = 1

# store results
results_Cv = []

# --- Loop over N and DL ---
for N in N_values:
    for DL in DL_values:
        print(f"Running Cv simulation: N={N}, L={L}, DL={DL}")
        out = run_ensemble(
            N=N, L=L, DL=DL, DL_kind=DL_kind,
            J=1.0, B=0.0, mu=1.0,
            Temperature=Temperature,
            equil_MCS_per_spin=equil_MCS_per_spin,
            sample_configs=sample_configs,
            Ns_between_samples=Ns_between_samples,
            realizations=realizations
        )
        results_Cv.append({
            "N": N, "DL": DL,
            "Temperature": Temperature,
            "Cv": out["Cv_avg"]
        })


In [ ]:
# Marker map as in susceptibility plot
# #marker_map = {
#     (100, 0.0): 'v',    # ▼
#     (50,  0.0): 'x',    # ×
#     (100, 2/3): '*',    # ★
#     (50,  2/3): 's',    # □
#     (100, 1.0): 'd',    # ◇
#     (50,  1.0): '^',    # △
# }

plt.figure(figsize=(8,6))

for r in results_Cv:
    N, DL = r["N"], r["DL"]
    plt.plot(
        r["Temperature"], r["Cv"],
        #marker=marker_map[(N, DL)],
        linestyle='-',
        markersize=6,
        label=f"N={N}, ΔL={DL}"
    )

plt.xlabel(r"$k_B T / J$", fontsize=14)
plt.ylabel(r"$C_v$", fontsize=14)
plt.title(r"Specific heat per spin, $L=2$")
plt.legend(fontsize=10)
plt.xlim(2.2, 3.5)
plt.tight_layout()
plt.show()
#

Magnetization

In [ ]:
results = []  # list of dicts

for N in N_values:
    for L in L_values:
        # Type I
        for DL in DL_values_I:
            out = run_ensemble(N, L, DL, "I", J=1.0, B=0.0, mu=1.0,
                               Temperature=Temperature,
                               equil_MCS_per_spin=200,   # smaller for test
                               sample_configs=50,
                               Ns_between_samples=10,
                               realizations=1)
            results.append({
                "N": N, "L": L, "DL": DL, "type": "I",
                "Temperature": Temperature,
                "M": out["M_avg"], "Cv": out["Cv_avg"], "chi": out["chi_avg"]
            })

        # Type II
        for DL in DL_values_II:
            out = run_ensemble(N, L, DL, "II", J=1.0, B=0.0, mu=1.0,
                               Temperature=Temperature,
                               equil_MCS_per_spin=200,
                               sample_configs=50,
                               Ns_between_samples=10,
                               realizations=1)
            results.append({
                "N": N, "L": L, "DL": DL, "type": "II",
                "Temperature": Temperature,
                "M": out["M_avg"], "Cv": out["Cv_avg"], "chi": out["chi_avg"]
            })


In [ ]:
plt.figure(figsize=(8,6))
for r in results:
    if r["N"] == 25 and r["type"] == "I" and r["DL"] == 0.0:
        plt.plot(r["Temperature"], r["M"], label=f"L={r['L']}, DL={r['DL']} ({r['type']})")

plt.xlabel("Temperature T")
plt.ylabel("Magnetization per spin")
plt.title("Magnetization vs T for different L (DL=0, Type I)")
plt.legend()
plt.show()


Magnetic Susceptibility

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


L = 2
N_values = [15,25,40]
DL_values = [0.0, 2/3, 1.0]   # Type I roughness
DL_kind = "I"
Temperature = np.linspace(2, 3.5, 40)  # 50 points between 2.6 and 3.5

# MC parameters (as in paper)
equil_MCS_per_spin = 50000  # equilibration
sample_configs = 10000        # number of measured configs
Ns_between_samples = 20       # skip between measurements
realizations = 1               # 1 for DL<1

results_fig2 = []

for N in N_values:
    for DL in DL_values:
        print(f"Running simulation for N={N}, L={L}, DL={DL}")
        out = run_ensemble(
            N=N, L=L, DL=DL, DL_kind=DL_kind,
            J=1.0, B=0.0, mu=1.0,
            Temperature=Temperature,
            equil_MCS_per_spin=equil_MCS_per_spin,
            sample_configs=sample_configs,
            Ns_between_samples=Ns_between_samples,
            realizations=realizations
        )
        results_fig2.append({
            "N": N, "DL": DL,
            "Temperature": Temperature,
            "chi": out["chi_avg"]
        })


In [ ]:

plt.figure(figsize=(8,6))

# marker_map = {
#     (100, 0.0): 'v',    # ▼
#     (50,  0.0): 'x',    # ×
#     (100, 2/3): '*',    # ★
#     (50,  2/3): 's',    # □
#     (100, 1.0): 'd',    # ◇
#     (50,  1.0): '^',    # △
# }

for r in results_fig2:
    N, DL = r["N"], r["DL"]
    plt.plot(
        r["Temperature"], r["chi"],
        #marker=marker_map[(N, DL)],
        linestyle='-', markersize=6,
        label=f"N={N}, ΔL={DL}"
    )

plt.xlabel(r"$ k_bT / J $", fontsize=14)
plt.ylabel(r"$\chi$", fontsize=14)
plt.title(r"Magnetic susceptibility per spin, $L=2$")
plt.legend(fontsize=10)
plt.xlim(2.2, 3.5)
plt.ylim(0, 0.02)
plt.tight_layout()
plt.show()
